# Lecture 7: Docker and Containerization for Machine Learning

How to "read" this lecture notebook
<details>
<summary>click to expand</summary>

As you go through this notebook (or any notebook for this class), you will encounter new concepts and terminal commands that implement them -- just like you would see in a textbook. Of course, in a textbook, it's easy to read commands and an explanation of what they do and think that you understand it.
<br />

### Learn by doing
But this notebook is different from a textbook because it allows you to not just read the commands, but actually run them. **You can and should try running these commands on your own machine**. In fact, in many places throughout this notebook, you will be asked to run commands yourself. This is a form of "active reading" and the idea behind it is that we really learn by **doing**.
<br />

### Experiment freely
I encourage you to experiment with different Docker commands and options. Docker is very forgiving -- if something goes wrong, you can always remove containers and start fresh. **Add your own notes and try variations** of the commands you see here.
<br />

### Make this notebook your own
Make this notebook your own. Write your questions and thoughts. At the end of every reading notebook, I will ask the same set of questions to try to elicit your questions, reaction and feedback.
</details>

## Learning Objectives

By the end of this lecture, you will be able to:
- Understand containerization and why it matters for ML workflows
- Master Docker basics: images, containers, and Dockerfiles
- Build custom Docker images for ML applications
- Manage volumes for data persistence
- Map ports to expose applications running in containers
- Deploy containerized ML models

# 7.0 Prerequisites

Before we begin, make sure you have:
1. **Docker Desktop installed** on your machine (from Lecture 2)
2. Docker Desktop **running** (you should see the whale icon in your system tray/menu bar)

To verify Docker is working, open a terminal and run:

```bash
docker --version
```

You should see something like `Docker version 24.x.x` or similar.

# 7.1 Understanding Containers and Docker

<img alt="Morpheus loads a reality for Neo, just like a docker container" src="../images/L07_matrix_containers.png" width="900" style="display:block;">
<font size=2>Morpheus loads a reality for Neo in <i>The Matrix (1999)</i></font>

Remember *The Matrix*? The entire world that Neo lived in was actually a simulation -- a self-contained universe with its own rules, running on top of the "real" infrastructure.

**Docker containers are remarkably similar.** Each container is like its own little Matrix -- a complete, isolated universe with its own operating system, libraries, and applications. From inside the container, it *feels* like you're on a dedicated machine. But in reality, many of these "universes" can coexist on a single physical computer, each blissfully unaware of the others.

Let's explore why this matters and how it solves real problems in software development and data science.

## The "But it Works on My Machine" Problem

Have you ever had this experience?

- You write some Python code that works perfectly on your laptop
- You send it to a colleague, and it crashes on their machine
- They have a different Python version, missing libraries, or different system configurations

In data science and machine learning, this problem is even worse because:
- ML projects have **many dependencies** (numpy, pandas, scikit-learn, tensorflow, specific CUDA versions...)
- Training results should be **reproducible** across machines
- Moving from development to production is notoriously difficult

**Containers solve this problem** by packaging your code AND its entire environment together.

## What is a Container?

<img alt="Docker containers like shipping containers" src="../images/L07_docker_shipping_container.png" width="700" style="display:block;">
<font size=2>Just like shipping containers standardized global trade, software containers standardize software deployment</font>

A **container** is a lightweight, standalone package that includes:
- Your application code
- All its dependencies and libraries
- Runtime environment (Python interpreter, etc.)
- System tools and configurations

Think of a container as a **shipping container** for software. Before shipping containers, loading cargo onto ships was chaotic -- every item had different shapes, sizes, and handling requirements. Shipping containers provided a **standard interface** that worked everywhere.

Similarly, software containers provide a standard interface for deploying applications. If it runs in a container on your machine, it will run the same way on:
- Your colleague's laptop
- A cloud server (AWS, GCP, Azure)
- A production cluster

## Containers vs Virtual Machines

<img alt="L07_containers_vs_vms.png" src="../images/L07_container_vs_vm.png" style="display:block;">
<font size=2>Containers share the host OS kernel, making them much lighter than VMs</font>


You might be thinking: "Can't I just use a virtual machine (VM) to get reproducibility?"

Yes, but containers are **much more efficient**:

| Feature | Virtual Machine | Container |
|---------|----------------|----------|
| **Size** | GBs (full OS) | MBs (just your app) |
| **Startup time** | Minutes | Seconds |
| **Resource usage** | Heavy | Lightweight |
| **Isolation** | Complete (separate OS) | Process-level |
| **Runs** | On a hypervisor | On container runtime |

**Virtual machines** virtualize the entire hardware -- each VM runs its own operating system.

**Containers** virtualize the operating system -- they share the host's OS kernel but keep everything else isolated.

For ML workflows, containers are ideal because you can:
- Start them in seconds (not minutes)
- Run many containers on one machine
- Use them for quick experiments and iteration


## Docker Architecture


<img alt="L07_docker_architecture.png" src="../images/L07_docker_architecture.png" width="700" style="display:block;">
<font size=2>Images are blueprints, Containers are running instances, Daemon manages client requests. credit: bytebytego.com</font>

**Docker** is the most popular containerization platform. Let's understand its key components.


### Key Concepts

1. **Docker Image**: A read-only template (blueprint) for creating containers
   - Contains the OS, application, dependencies, and configurations
   - Like a **class** in OOP -- defines what a container should be

2. **Docker Container**: A running instance of an image
   - Like an **object** in OOP -- a specific instance created from the class
   - You can run multiple containers from the same image

3. **Dockerfile**: A text file with instructions to build an image
   - Like a recipe that Docker follows

4. **Docker Registry**: A storage and distribution system for images
   - **Docker Hub** is the most popular public registry
   - You can pull pre-built images or push your own

5. **Docker Daemon**: The background service that manages containers
   - This is what Docker Desktop starts for you

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

<b>The Image-Container Relationship</b>: Remember out cookie cutter/cookie analogy for Classes and Objects? Well the same is true of images and conatiners. An <b>image</b> is like a cookie cutter (the template), and a <b>container</b> is like a cookie (a specific instance). You can make many cookies from one cookie cutter, and each cookie can be decorated differently (have different data, state, etc.) while all starting from the same base shape.

  
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

## Why Containers for Machine Learning?

Containers are especially valuable for ML because:

1. **Reproducibility**: Package your model with exact library versions
2. **Portability**: Train on your laptop, deploy to the cloud
3. **Isolation**: Different projects can have conflicting dependencies
4. **Scalability**: Easily scale across multiple machines
5. **Collaboration**: Share exact environments with teammates

Major ML platforms all support containerization:
- AWS SageMaker uses Docker containers
- Google AI Platform uses Docker containers
- Kubernetes (for scaling) orchestrates containers

<!-- Start Exercise 7.1 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> In Your Own Words: What Problem Do Containers Solve? </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
In your own words, explain what problem containers solve and why this is particularly important for machine learning projects. Try to give a concrete example of a problem you might like to solve as an ML product manager.
</div>

ENTER YOUR EXPLANATION HERE.

<hr/>
<!-- End Exercise 7.1 -->

# 7.2 Your First Docker Commands

<img alt="L07_neo_knows_kung_fu.png" src="../images/L07_neo_knows_kungfu.png" width="1000" style="display:block;">
<font size=2>"I know Kung Fu." -- Neo, after having  martial arts uploaded directly into his brain in <i>The Matrix (1999)</i></font>

Remember when Neo had kung fu uploaded directly into his mind? One moment he knew nothing about martial arts, and seconds later he was a master. **Docker gives you a similar superpower.**

With a single command, you can instantly "download" entire software environments -- a complete Python data science stack, a configured database server, a machine learning framework with GPU support. Someone else did all the hard work of installing and configuring everything. You just pull their image and suddenly... *you know kung fu*.

Let's see how this works.

## Running Your First Container

Let's start by running a simple container. Open your terminal and type:

```bash
docker run hello-world
```

This command does several things:
1. Looks for an image called `hello-world` on your machine
2. If not found, **pulls** it from Docker Hub
3. Creates a **container** from that image
4. **Runs** the container (which prints a message)
5. **Exits** when done

You should see output like:
```
Hello from Docker!
This message shows that your installation appears to be working correctly.
...
```

## Understanding `docker run`

The `docker run` command is your most-used Docker command. Its basic syntax is:

```bash
docker run [OPTIONS] IMAGE [COMMAND]
```

Let's try a more interesting example -- running an Ubuntu container:

```bash
docker run -it ubuntu bash
```

Breaking this down:
- `docker run` - create and start a container
- `-it` - interactive mode with a terminal (this is actually two flags combined: `-i` for interactive and `-t` for terminal)
- `ubuntu` - the image to use (official Ubuntu image from Docker Hub)
- `bash` - the command to run inside the container

You're now **inside a running Ubuntu container**! Notice the prompt changed. Try some commands:

```bash
cat /etc/os-release   # See what OS you're running
whoami                 # You're root!
ls /                   # Browse the file system
exit                   # Leave the container
```

When you type `exit`, the container stops because its main process (bash) ended.

## Listing Containers and Images

After running some containers, let's see what we have:

**List running containers:**
```bash
docker ps
```

If you've exited all containers, this will be empty.

**List ALL containers (including stopped ones):**
```bash
docker ps -a
```

Now you'll see the containers you've run. Notice each has:
- **CONTAINER ID**: Unique identifier
- **IMAGE**: What image it was created from
- **COMMAND**: The command that was run
- **STATUS**: Running, Exited, etc.
- **NAMES**: Docker assigns random names if you don't specify one

**List downloaded images:**
```bash
docker images
```

You should see `hello-world` and `ubuntu` images.

## Pulling Images

You can download images without running them using `docker pull`:

```bash
docker pull python:3.11
```

This pulls the official Python 3.11 image. The `:3.11` is called a **tag** and specifies a particular version. Some common conventions:
- `python:latest` - latest version (default if no tag specified)
- `python:3.11` - specific version
- `python:3.11-slim` - smaller image without extra tools
- `python:3.11-alpine` - even smaller, based on Alpine Linux

## Running Detached Containers

Sometimes you want a container to run in the background. Use the `-d` flag:

```bash
docker run -d --name my-python python:3.11 sleep 3600
```

Breaking this down:
- `-d` - **detached** mode (runs in background)
- `--name my-python` - give it a friendly name (instead of random one)
- `python:3.11` - the image
- `sleep 3600` - command to run (sleep for 1 hour)

Now check running containers:
```bash
docker ps
```

You should see your container running!

## Container Lifecycle Commands

A docker container will run until its main program is complete or until it is stopped. Here are the essential commands for managing containers:

**Stop a running container:**
```bash
docker stop my-python
```

**Start a stopped container:**
```bash
docker start my-python
```

**Restart a container:**
```bash
docker restart my-python
```

**Remove a stopped container:**
```bash
docker rm my-python
```

**Force remove a running container:**
```bash
docker rm -f my-python
```

**Remove an image:**
```bash
docker rmi hello-world
```



<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
  display: flex;
  align-items: flex-start;
">
  <div style="
    font-size: 60pt;
    line-height: 1;
    width: 48px;
    display: flex;
    align-items: flex-start;
  ">
    🛠️
  </div>
<div style="padding: 0px; margin:0 0 0 60px;">

Existing **docker images** can be used directly or built upon to do super useful stuff! For example, on my home server, I have a container for a **my website** (built off nginx image), a **plex server**, a **local llm** (we'll  talk more about this later), and many other services. Here's an example of a script for pulling the latest version of my website from github, creating a small container and running it:

<font size=5>`updatewww.sh`</font>
<hr />

```bash
docker stop dylantwalkerdotcom 
docker wait dylantwalkerdotcom
docker rm dylantwalkerdotcom
docker pull ghcr.io/dylanwalker/dylantwalkerdotcom:latest
docker create  --name=dylantwalkerdotcom -e PUID=1000 -e PGID=1000 -p 80:80 --restart=unless-stopped ghcr.io/dylanwalker/dylantwalkerdotcom:latest
docker start dylantwalkerdotcom
```

<hr />
</div>
</div>

## Executing Commands in Running Containers

You can run commands inside a running container with `docker exec`:

First, start a container in the background. This one will just write the datetime tick/tock every second:
```bash
docker run -d --name my-container ubuntu bash -c 'while true; do echo "$(date) tick"; sleep 1; echo "$(date) tock"; sleep 1; done'
```

Now execute commands in it:
```bash
docker exec my-container cat /etc/os-release
```

Or get an interactive shell:
```bash
docker exec -it my-container bash
```

This is extremely useful for debugging or exploring what's inside a container. When you exit this shell, the container keeps running (unlike with `docker run -it`).

## Viewing Container Logs

Containers capture output from their **main process** (stdout/stderr of PID1). View logs with:

```bash
docker logs my-container
```

Follow logs in real-time (like `tail -f`):
```bash
docker logs -f my-container
```

Show only last 10 lines:
```bash
docker logs --tail 10 my-container
```



## Inspecting Containers

Get detailed information about a container:

```bash
docker inspect my-container
```

This returns a JSON blob with everything: network settings, volumes, environment variables, etc.

Filter for specific info:
```bash
docker inspect --format '{{.State.Status}}' my-container
docker inspect --format '{{.NetworkSettings.IPAddress}}' my-container
```

Ok, let's make sure we clean up any containers we just created that we don't want using `docker ps -a` and `docker rm -f <containerid/container_name>`.

<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Practice: Container Lifecycle </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Open your terminal and practice the container lifecycle:
<ol>
<li>Pull the <code>alpine</code> image (a tiny Linux distribution)</li>
<li>Run a container named <code>my-alpine</code> with the command <code>sleep 1000</code> in detached mode</li>
<li>Verify it's running with <code>docker ps</code></li>
<li>Stop the container</li>
<li>Restart the container</li>
<li>Execute the command <code>cat /etc/os-release</code> inside it</li>
<li>Remove the container</li>
</ol>
</div>

<hr/>

# 7.3 Docker Images and Dockerfiles

<img alt="They Live" src="../images/L07_they_live.png" width="700" style="display:block;">
<font size=2>These special sunglasses show Nada the truth in <i>They Live (1988)</i></font>

Before we can build our own Docker images, we need to learn about the format of Dockerfiles, which will server as our blueprint or recipe. These file will contain everything we need to build our own custom images, which is where the real power of Docker comes in.

## What is a Dockerfile?

<img alt="A dockerfile is a recipe" src="../images/L07_dockerfile_recipe.png" width="500" style="display:block;">
<font size=2>A Dockerfile is a recipe that Docker follows to build your image.  credit: Soroosh Nazem</font>

While you can use pre-built images from Docker Hub, the real power of Docker comes from **building your own images**.

A **Dockerfile** is a text file containing instructions to build an image. It's like a recipe:



Each instruction in a Dockerfile creates a **layer** in the image. Layers are cached, which makes subsequent builds faster.

## Essential Dockerfile Instructions

Let's start by looking at an example (we won't actually build this image):

**Dockerfile**:

```dockerfile
# FROM Specifies the starting docker image
FROM python:3.11

# RUN Runs terminal commands during the build process (these may install software in the image, create files, etc)
RUN apt-get update && apt-get install -y nano
RUN pip install pandas numpy scikit-learn            

# WORKDIR Sets the working directory for subsequent commands
WORKDIR /app

# COPY Copies files from your local machine into the image
COPY readme.md .
COPY app.py .

# ENV Sets environment variables
ENV PYTHONUNBUFFERED=1
ENV MODEL_PATH=/app/models

# EXPOSE Documents that you intent to have the following ports exposed (to accomplish this, you will have to map these ports to HOST when you run the container, though)
EXPOSE 8000

# CMD Specifies the default command to run when the container starts, along with any arguments it needs, in this "list format": [arv0, argv1, ...]
CMD ["python", "app.py"]

# Instead of the above, we could have written:
ENTRYPOINT["python"]
CMD["app.py"]
# This ENTRYPOINT/CMD method is preferred if you want to say "ENTRYPOINT is the program", but feeel free to adjust its argument ("app.py")
# The CMD only method is more casual, like saying "By default, we'll run 'python app.py', but feel free to do what you want"
```


## Your First Dockerfile

Let's create a simple Dockerfile. Create a new directory and add these files:

**Directory structure:**
```
my-docker-project/
├── Dockerfile
└── app.py
```

**app.py:**
```python
import pandas as pd

# Create a simple DataFrame
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'score': [85, 92, 78]
})

print("Hello from Docker!")
print(f"\nPandas version: {pd.__version__}")
print(f"\nDataFrame:\n{df}")
print(f"\nMean score: {df['score'].mean():.2f}")
```

**Dockerfile:**
```dockerfile
# Start from Python 3.11 base image
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Install pandas
RUN pip install pandas

# Copy our application
COPY app.py .

# Run the application when container starts
CMD ["python", "app.py"]
```

Let's make the directory structure on our local machine for this project and copy the content above into `Dockerfile` and `app.py`.

## Building the Image

We can build our image from this recipe with `docker build`:

```bash
cd my-docker-project
docker build -t my-pandas-app .
```

Breaking this down:
- `docker build` - build an image from a Dockerfile
- `-t my-pandas-app` - tag (name) the image as "my-pandas-app"
- `.` - use the current directory (where Dockerfile is located)

Watch as Docker:
1. Downloads the base image (if needed)
2. Runs each instruction in the Dockerfile
3. Creates layers for each step
4. Tags the final result

Verify the image was created:
```bash
docker images | grep my-pandas-app
```

## Running Your Custom Image

Now run a container from your image:

```bash
docker run --name my-pandas-app-container my-pandas-app
```

You should see the output from your Python script!

We can see this output at a later time using docker logs: `docker logs my-pandas-app-container`

The key insight: **anyone with this Dockerfile can build the exact same image and get the exact same results**. No more "but it works on my machine"!

## Layer Caching

Docker caches each layer. If you rebuild:

```bash
docker build -t my-pandas-app .
```

You'll see "CACHED" for unchanged layers. This makes rebuilds fast!

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

<b>Best Practice</b>: Order your Dockerfile instructions from least-frequently-changed to most-frequently-changed. Put <code>pip install</code> before <code>COPY</code> of your source code. That way, changing your code doesn't invalidate the cached dependencies layer.

  
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Practice: Build Your Own Image </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Create a directory called <code>hello-docker</code> and build an image that:
<ol>
<li>Uses <code>ubuntu</code> as the base image</li>
<li>Installs <code>nano</code> and <code>python3</code> using <code>apt-get</code></li>
<li>Installs <code>pandas</code> using pip</li>
<li>Sets an ENTRYPOINT that keeps it running (hint: use <code>sleep infinity</code> or <code>tail -f /dev/null</code>)</li>
</ol>
Build it with <code>docker build -t my-ubuntu .</code> and verify the image exists.
</div>

```Dockerfile

PASTE YOUR DOCKERFILE CONTENT HERE

```

<hr/>

# 7.4 Volumes and Data Persistence

<img alt="Say Anything Boombox Scene" src="../images/L07_say_anything.png" width="800" style="display:block;">
<font size=2> Lloyd Dobler demonstrates persistence in <i>Say Anything (1989)</i></font>

We want our data to persist (stick around) even after our containers are done. But by design, containers don't behave that way.

## Containers Are Ephemeral

By default, containers are **ephemeral** -- any data created inside disappears when the container is removed:

```bash
# Create a file inside a container
docker run -it --name test-container ubuntu bash
# Inside the container:
echo "Hello" > /myfile.txt
cat /myfile.txt  # File exists!
exit

# Remove and recreate
docker rm test-container
docker run -it --name test-container ubuntu bash
# Inside the container:
cat /myfile.txt  # File is gone!
```

For ML workflows, this is problematic:
- Training data shouldn't be inside the container
- Model outputs need to persist
- Logs should survive container restarts


There are two solutions to this problem: **bind mounts** and **named volumes**:

<img alt="Bind mount vs named volumes in docker" src="../images/L07_bind_mount_vs_named_volume.png" width="800" style="display:block;">


- Bind mounts attach or "bind" directories (folders) on your host machine to directories inside containers
- Named volumes are like persistent "virtual disks" that Docker itself manages. Any container can mount these "disks"

## Bind Mounts (sharing host directories)

A **bind mount** maps a directory on your host machine to a directory inside the container. They're good for data that you need to access from within the container and directly from your host machine.

To mount a directory
```bash
docker run -v /path/on/host:/path/in/container image
```

**Example:** Let's share a data directory:

```bash
# Create a directory with some data
mkdir ~/my-data
echo "Important data" > ~/my-data/data.txt

# Run container with bind mount
docker run -it -v ~/my-data:/data ubuntu bash
```

Inside the container:
```bash
cat /data/data.txt      # See the host file!
echo "From container" > /data/from_container.txt
exit
```

Back on your host:
```bash
cat ~/my-data/from_container.txt  # File created by container!
```

Changes go **both ways** -- the container can read and write to your host files.

### Practical Example: ML Data Pipeline

Here's a common pattern for ML:

```bash
docker run -v ~/datasets:/data \
           -v ~/models:/output \
           my-ml-image python train.py
```

This mounts:
- Your `datasets` folder to `/data` (for reading training data)
- Your `models` folder to `/output` (for saving trained models)

The training script inside reads from `/data` and saves to `/output`. After the container exits, your trained model is safely on your host machine!

## Named Volumes (like persistent virtual disks)

Docker can also manage volumes for you with named volumes.

**Named volumes** are managed by Docker and stored in a Docker-controlled location. They're good for:
- Data you don't need to access directly from the host
- Database storage
- Sharing data between containers

```bash
# Create a named volume
docker volume create mydata

# Use it in a container
docker run -v mydata:/data ubuntu bash
```

List volumes:
```bash
docker volume ls
```



<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Practice: Persisting Data </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Using your <code>my-ubuntu</code> image from earlier:
<ol>
<li>Create a directory on your host called <code>docker-test</code></li>
<li>Run a container with that directory mounted to <code>/workspace</code></li>
<li>Use <code>docker exec</code> to shell into the container</li>
<li>Create a file using <code>nano</code> in <code>/workspace</code></li>
<li>Exit and verify the file exists on your host machine</li>
</ol>
</div>

<hr/>

# 7.5 Networking and Port Mapping

<img alt="Sneakers Hacking" src="../images/L07_sneakers_hacking.png" width="900" style="display:block;">
<font size=2>Whistler hacks the network in <i>Sneakers (1992)</i>, the <b>greatest hacking movie of all time</b></font>

Just like any computer, containers have a way of accessing the network, but in order to use it, you will need to determine how the container is wired up to the host machine's network. 

## Container Networking Basics

Containers are isolated by default -- they can't be accessed from outside unless you explicitly allow it.

To expose a web application or API running inside a container, you need **port mapping**. Port mapping connects a port on your host to a port inside the container.

<img alt="L07_port_mapping.png" src="../images/L07_docker_port_mapping.png" width="600" style="display:block;">
<font size=2>credit: linuxhandbook.com</font>

Let's have look at how we can specify port mapping when we run a container, and then how we can use it to access a web application running inside a container.

## The `-p` Flag

Port mapping is easy, just use `-p` to map ports from the host to the container with the following syntax:

```bash
docker run -p HOST_PORT:CONTAINER_PORT image
```

**Example:** Run a Jupyter notebook server:

```bash
docker run -p 8888:8888 jupyter/scipy-notebook
```

This maps:
- Port 8888 on your host machine
- To port 8888 inside the container (where Jupyter runs)

Now open `http://localhost:8888` in your browser!

You can use different ports:
```bash
docker run -p 9999:8888 jupyter/scipy-notebook
```

Now access Jupyter at `http://localhost:9999`.


## Running a Web Application

Let's create a tiny **FastAPI** app and containerize it. We'll use **uvicorn** to serve the application.

FastAPI is a modern, high-performance web framework that's becoming increasingly popular for ML APIs. It offers automatic documentation, type validation, and async support.

Uvicorn implements a fast ASGI (Asynchronous Server Gateway Interface) server in Python.

**Directory structure:**
```
fastapi-app/
├── Dockerfile
├── requirements.txt
└── app.py
```

**requirements.txt:**
```
fastapi
uvicorn[standard]
```

**app.py:**
```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Hello from Docker!", "status": "running"}

@app.get("/predict")
def predict():
    # In a real app, this would run your ML model
    return {"prediction": 0.87, "confidence": "high"}
```

**Dockerfile:**
```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

Build and run:
```bash
docker build -t my-fastapi-app .
docker run --rm -p 8000:8000 my-fastapi-app
```

Now visit:
- `http://localhost:8000/` - Hello message
- `http://localhost:8000/predict` - Prediction endpoint
- `http://localhost:8000/docs` - **Automatic interactive API documentation!**

FastAPI automatically generates Swagger UI documentation at `/docs` -- this is incredibly useful for ML APIs where you want collaborators to easily test your model endpoints.

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

<b>Why <code>host='0.0.0.0'</code>?</b>: When running Flask (or any server) inside a container, you must bind to <code>0.0.0.0</code> (all interfaces), not <code>localhost</code> or <code>127.0.0.1</code>. The latter only accepts connections from inside the container, but you want to accept connections forwarded from the host.

  
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

# 7.6 Docker for ML Workflows

<img alt="The Martian Science" src="../images/L07_martian_science.png" width="900" style="display:block;">
<font size=2>"In the face of overwhelming odds, I'm left with only one option. I'm gonna have to science the sh*t out of this" - Mark Watney, <i>The Martian (2015)</i></font>

We'll put everything we've learned so far together into a concrete example of how Docker can be used in an ML workflow for our next lab assignment. For now, here's some high-level examples:


## A Typical ML Dockerfile

Here's a more realistic Dockerfile for an ML project:


```dockerfile
# Use an official Python runtime as base
FROM python:3.11-slim

# Set environment variables
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

# Set work directory
WORKDIR /app

# Install system dependencies (if needed)
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY src/ ./src/
COPY models/ ./models/

# Expose port for API
EXPOSE 8000

# Run the application
CMD ["python", "src/serve.py"]
```

Key patterns:
- **Multi-line ENV** for cleaner environment setup
- **Clean up apt cache** to reduce image size
- **Copy requirements first** to leverage layer caching
- **Use `--no-cache-dir`** with pip to reduce image size

## GPU Support

For deep learning, you often need GPU access. The trick is to use a container with CUDA drivers already installed (though you can build your own image that installs CUDA). When running a CUDA-supported container, you should add the arguments `--gpus all` to your docker run command like this:

```bash
docker run --gpus all nvidia/cuda:11.8-base nvidia-smi
```

This makes all gpus on the host machine available to your container.

**note**:  `nvidia-smi` is Nvidia's system management interface command line tool. You can also run it when shelled into a container to check that it sees your GPUs, see their utilization, etc.

Requirements:
- NVIDIA GPU on host
- NVIDIA drivers installed on host
- NVIDIA Container Toolkit installed

For example, you can run the default pytorch container likes this:
```bash
docker run --gpus all pytorch/pytorch:latest python -c "import torch; print(torch.cuda.is_available())"
```

We won't go deep into GPU Docker in this lecture (its likely that your laptop doesn't have a capable enough GPU), but know that it's possible and commonly used for training deep learning models.

## Docker Compose

When your application needs multiple containers (e.g., ML API + database + message queue), **Docker Compose** helps manage them together.

A `docker-compose.yml` file:

```yaml
version: '3.8'

services:
  api:
    build: .
    ports:
      - "8000:8000"
    volumes:
      - ./data:/data
    
  redis:
    image: redis:alpine
    ports:
      - "6379:6379"
```

Start everything with:
```bash
docker compose up
```

Stop everything:
```bash
docker compose down
```

Docker Compose is beyond the scope of this lecture, but it's worth knowing it exists for more complex deployments.

## Pushing to Docker Hub

Share your images by pushing to Docker Hub:

```bash
# Log in to Docker Hub
docker login

# Tag your image with your username and a version number
docker tag my-flask-api yourusername/my-flask-api:v1.0

# Push to Docker Hub
docker push yourusername/my-flask-api:v1.0
```

Now anyone can pull and run your image:
```bash
docker pull yourusername/my-flask-api:v1.0
docker run -p 5000:5000 yourusername/my-flask-api:v1.0
```

# 7.7 Quick Reference: Docker Commands Cheat Sheet

## Images

| Command | Description |
|---------|-------------|
| `docker images` | List images |
| `docker pull IMAGE` | Download an image |
| `docker build -t NAME .` | Build image from Dockerfile |
| `docker rmi IMAGE` | Remove an image |
| `docker tag IMAGE NEW_NAME` | Tag an image |
| `docker push IMAGE` | Push to registry |

## Containers

| Command | Description |
|---------|-------------|
| `docker run IMAGE` | Create and start container |
| `docker run -d IMAGE` | Run in background |
| `docker run -it IMAGE bash` | Interactive terminal |
| `docker run -p 8080:80 IMAGE` | Map port 8080→80 |
| `docker run -v /host:/container IMAGE` | Mount volume |
| `docker run --name NAME IMAGE` | Name the container |
| `docker ps` | List running containers |
| `docker ps -a` | List all containers |
| `docker stop CONTAINER` | Stop container |
| `docker start CONTAINER` | Start stopped container |
| `docker restart CONTAINER` | Restart container |
| `docker rm CONTAINER` | Remove container |
| `docker rm -f CONTAINER` | Force remove |
| `docker exec -it CONTAINER bash` | Shell into container |
| `docker logs CONTAINER` | View logs |
| `docker logs -f CONTAINER` | Follow logs |
| `docker inspect CONTAINER` | Detailed info |

## Volumes

| Command | Description |
|---------|-------------|
| `docker volume create NAME` | Create named volume |
| `docker volume ls` | List volumes |
| `docker volume rm NAME` | Remove volume |

## Cleanup

| Command | Description |
|---------|-------------|
| `docker system prune` | Remove unused data |
| `docker container prune` | Remove stopped containers |
| `docker image prune` | Remove dangling images |

# 7.8 Summary and Key Takeaways

## What We Learned

1. **Containers solve the "but it works on my machine" problem** by packaging code AND environment together

2. **Docker terminology:**
   - **Image** = blueprint (class)
   - **Container** = running instance (object)
   - **Dockerfile** = recipe to build an image
   - **Registry** = storage for images (Docker Hub)

3. **Essential commands:**
   - `docker run` - create and start containers
   - `docker build` - build images from Dockerfiles
   - `docker exec` - run commands in containers
   - `docker ps` - list containers

4. **Data persistence:**
   - Containers are ephemeral by default
   - Use bind mounts (`-v /host/path:/container/path`) to persist data

5. **Networking:**
   - Use `-p HOST:CONTAINER` to expose ports
   - Bind to `0.0.0.0` inside containers for external access

6. **ML benefits:**
   - Reproducibility across environments
   - Easy deployment to cloud platforms
   - Isolation between projects

## Why This Matters for ML

- **Reproducibility**: Critical for scientific ML work
- **Deployment**: Most ML deployment platforms use containers
- **Collaboration**: Share exact environments with your team
- **Scaling**: Containers are the foundation of Kubernetes and cloud ML

# Your Turn: Questions, Reactions, and Feedback


Before our next class, think about:

1. What concepts were new to you?
2. What do you want more practice with?
3. Can you think of a project where containers would have helped you?
4. Any questions or points of confusion?

**Write your thoughts below:**

*[Your reflections here]*